# Feelody · 감정 재순위화
GPU 런타임을 선택하고 압축 패키지 안의 colab_inputs.zip을 업로드합니다. 원문이 포함된 입력은 개인 연구용입니다.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io
uploaded = files.upload()
assert len(uploaded) == 1, 'colab_inputs.zip 하나를 선택하세요.'
with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
    z.extractall('/content/feelody_research')
%cd /content/feelody_research

## 기존 평가 재현
새로 채점한 xlsx가 있으면 replace_workbook을 True로 바꾼 뒤 업로드하세요.

In [ ]:
replace_workbook = False
if replace_workbook:
    updated = files.upload()
    assert len(updated) == 1
    Path('Feelody_blind.xlsx').write_bytes(next(iter(updated.values())))
from evaluation.score_fusion import score
score('lexical_run.json','qrels.csv','Feelody_blind.xlsx')

## 모델 추론
실제 torch 버전과 모델 revision을 결과에 기록합니다. 설치 후 런타임 재시작을 요구하면 재시작한 다음 아래 추론 셀부터 진행하세요.

In [ ]:
%pip -q install transformers==4.57.1 huggingface-hub==0.36.2 tokenizers==0.22.2 sentencepiece scipy openpyxl

In [ ]:
%cd /content/feelody_research
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
from evaluation.fusion import run
run('feelody_paired10_gpu.zip','fusion_results.json')

In [ ]:
from evaluation.score_fusion import score
from google.colab import files
score('fusion_results.json','qrels.csv','Feelody_blind.xlsx')
files.download('fusion_results.json')